In [ ]:
from dotenv import load_dotenv
import os

from azure.identity import DefaultAzureCredential
from openai import AzureOpenAI

import fitz 
import numpy as np
from paddleocr import PaddleOCR

load_dotenv()

In [ ]:
def ocr_document(pdf_path: str, dpi: int = 300, lang: str = "en", conf_min: float = 0.5, native_min_chars: int = 40):
    """
    OCR a single PDF document (path).
    - Extracts native text if available.
    - Falls back to PaddleOCR for scanned pages.
    Returns: list of dicts [{page, method, text}]
    """
    # Initialize PaddleOCR with correct parameters for v3.x
    ocr = PaddleOCR(use_textline_orientation=True, lang=lang)

    def page_to_image(page):
        zoom = dpi / 72
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
        if pix.n == 4:
            img = img[:, :, :3]
        return img

    doc = fitz.open(pdf_path)
    pages = []

    for i, page in enumerate(doc, start=1):
        native_text = (page.get_text("text") or "").strip()

        if len(native_text) >= native_min_chars:
            text = native_text
            method = "native"
        else:
            img = page_to_image(page)
            result = ocr.ocr(img)
            lines = []
            if result and result[0]:
                for line in result[0]:
                    if line and len(line) >= 2:
                        # Structure: [[bbox], (text, confidence)]
                        txt, conf = line[1][0], line[1][1]
                        if conf >= conf_min:
                            lines.append(txt.strip())
            text = "\n".join(lines).strip()
            method = "ocr"

        pages.append({"page": i, "method": method, "text": text})

    doc.close()
    return pages

In [ ]:
data = ocr_document("data/fortimo-led-linear-dig.pdf", lang="en")

In [ ]:
# Format and save extracted text to a file

output_dir = "data_output"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "extracted_text.txt")

with open(output_file, "w", encoding="utf-8") as f:
    for page_data in data:
        f.write(f"{'='*80}\n")
        f.write(f"PAGE {page_data['page']} (Extraction method: {page_data['method']})\n")
        f.write(f"{'='*80}\n\n")
        f.write(page_data['text'])
        f.write(f"\n\n")

print(f"✓ Extracted text saved to: {output_file}")
print(f"  Total pages: {len(data)}")
print(f"  Total characters: {sum(len(p['text']) for p in data)}")

In [ ]:
# Open the file in VS Code
from IPython.display import display, FileLink

# Display clickable link
display(FileLink(output_file))

In [ ]:
credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default")

azure_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=token.token,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

def normalize(v):
    v = np.array(v, dtype="float32")
    return v / np.linalg.norm(v)

def get_text_embedding(text, normalizeEmbedding=True, model="text-embedding-3-large"):
    emb = azure_client.embeddings.create(input=[text], model=model).data[0].embedding
    if normalizeEmbedding:
        emb = normalize(emb)
    return emb

In [ ]:
import glob

# Process all PDFs in data folder

all_results = []

pdf_files = glob.glob("data/*.pdf")

print(f"Found {len(pdf_files)} PDF files")

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path}: {pdf_files.index(pdf_path)+1}/{len(pdf_files)}")
    pages = ocr_document(pdf_path, lang="en")
    all_results.append({
        "file": pdf_path,
        "pages": pages
    })
    print(f"  ✓ Extracted {len(pages)} pages")

print(f"\n✓ Total files processed: {len(all_results)}")